# 02 Cleaning

Use this notebook to clean the raw data, document your transformations, and export a processed file to `data/processed/`.

In [4]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Load the raw dataset
# Make sure the path matches your GitHub structure
df_raw = pd.read_csv('../data/raw/RuralCreditData.csv')

print(f"Dataset loaded successfully. Shape: {df_raw.shape}")
df_raw.head()

Dataset loaded successfully. Shape: (40000, 21)


,Id,city,age,sex,social_class,primary_business,secondary_business,annual_income,monthly_expenses,old_dependents,...,home_ownership,type_of_house,occupants_count,house_area,sanitary_availability,water_availabity,loan_purpose,loan_tenure,loan_installments,loan_amount
0,1,Dhanbad,22,F,Mochi,Tailoring,Others,36000.0,5000.0,0,...,1.0,R,4,70.0,1.0,0.5,Apparels,12,12,5000.0
1,2,Manjapra,21,F,OBC,Tailoring,none,94000.0,3600.0,1,...,1.0,T1,4,80.0,1.0,0.5,Apparels,12,50,7500.0
2,3,Dhanbad,24,M,Nai,Beauty salon,Others,48000.0,4000.0,0,...,1.0,T1,4,50.0,1.0,0.5,Beauty Salon,12,12,5000.0
3,4,NaN,26,F,OBC,Tailoring,none,7000.0,5000.0,0,...,1.0,T1,5,50.0,1.0,0.5,Apparels,12,50,7500.0
4,5,Nuapada,23,F,OBC,General store,Agriculture,36000.0,3500.0,0,...,1.0,T1,1,112.0,1.0,0.5,Retail Store,12,12,5000.0


In [5]:
# 1. Rename the misspelled column 'water_availabity' to 'water_availability'
df = df_raw.copy()
df.rename(columns={'water_availabity': 'water_availability'}, inplace=True)

# 2. Drop the 'id' column as it is not useful for business analysis
df.drop(columns=['Id'], inplace=True)

print("Structural cleaning complete: Column renamed and ID dropped.")

Structural cleaning complete: Column renamed and ID dropped.


In [ ]:
# Add project-specific cleaning steps below.
# Example: parse dates, fill null values, standardize category labels, remove outliers.

In [6]:
# Standardizing the 'social_class' column
# We found 'Mochi' and 'Mouchi' are the same class.
df['social_class'] = df['social_class'].replace('Mouchi', 'Mochi')

# Standardizing 'sex' to be consistent (Ensure all are 'M' or 'F')
df['sex'] = df['sex'].str.upper().str.strip()

print("Standardization complete: Fixed typos in social_class and sex.")

Standardization complete: Fixed typos in social_class and sex.


In [7]:
# 1. Handling Missing Values
# Fill missing numerical values with the Median (better than mean for skewed data)
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill missing categorical values with 'Unknown'
cat_cols = df.select_dtypes(include=['object']).columns
df[cat_cols] = df[cat_cols].fillna('Unknown')

# 2. Outlier Treatment (The 3,600 vs 36,000 issue)
# We will cap extreme values using the 95th percentile to avoid distorting the analysis
q_limit = df['annual_income'].quantile(0.95)
df['annual_income'] = np.where(df['annual_income'] > q_limit, q_limit, df['annual_income'])

print("Cleaning complete: Missing values handled and outliers capped.")

Cleaning complete: Missing values handled and outliers capped.


/var/folders/2p/dx176pz54kz6bzdn9qq091940000gn/T/ipykernel_5315/1879834967.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


In [8]:
# 1. Create 'Disposable Annual Income'
# Annual Income minus total yearly expenses
df['disposable_income'] = df['annual_income'] - (df['monthly_expenses'] * 12)

# 2. Create 'Debt-to-Income Ratio'
# How much of their annual income is the loan?
df['loan_to_income_ratio'] = (df['loan_amount'] / df['annual_income']) * 100

# 3. Create 'Total Dependents'
df['total_dependents'] = df['old_dependents'] + df['young_dependents']

print("Feature Engineering complete: Added disposable_income, loan_to_income_ratio, and total_dependents.")

Feature Engineering complete: Added disposable_income, loan_to_income_ratio, and total_dependents.


In [9]:
# Randomly sample 10,000 rows to keep the dataset manageable for Tableau
df_processed = df.sample(n=10000, random_state=42)

# Save the cleaned dataset to the processed folder
df_processed.to_csv('../data/processed/RuralCreditData_processed.csv', index=False)

print(f"Final dataset saved to data/processed/. Final Shape: {df_processed.shape}")

Final dataset saved to data/processed/. Final Shape: (10000, 23)
